In [1]:
from mistralai.client import Mistral
import os
import pandas as pd 
import faiss
import numpy as np
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time


In [2]:
# Initialiser l'encodeur (utilisez le modèle approprié)
encoding = tiktoken.get_encoding("cl100k_base")  # Encodeur standard pour la plupart des modèles

# Calculer les tokens pour une description
def count_tokens(text):
    return len(encoding.encode(text))

In [3]:
# Load file
source_data = pd.read_json('../data/evenements-publics-openagenda_26.json')

print(f'Nombre de lignes avant filtrage {source_data.shape[0]}')

# Filtrer les lignes de source_data avec des descriptions vide ou avec uniquement des espaces
source_data = source_data[source_data["longdescription_fr"].apply(lambda x: isinstance(x, str) and bool(x.strip()))]

print(f'Nombre de lignes après filtrage {source_data.shape[0]}')

long_desc_list = source_data["longdescription_fr"].tolist()

print(f'Nombre de lignes dans long_desc_list {len(long_desc_list)}')


Nombre de lignes avant filtrage 6925
Nombre de lignes après filtrage 6628
Nombre de lignes dans long_desc_list 6628


In [14]:
# Filtrer uniquement des événements avec des dates de début à mai 2026.
# Convert both dates to UTC timezone for comparison
current_date = pd.Timestamp.now(tz='UTC')
print(f"Date actuelle de référence : {current_date}")

# Convertir les colonnes de dates en datetime AVANT le filtrage
source_data["firstdate_begin"] = pd.to_datetime(source_data["firstdate_begin"], utc=True)
source_data["firstdate_end"] = pd.to_datetime(source_data["firstdate_end"], utc=True)
source_data["lastdate_begin"] = pd.to_datetime(source_data["lastdate_begin"], utc=True)
source_data["lastdate_end"] = pd.to_datetime(source_data["lastdate_end"], utc=True)

# Filtrer les événements dont la date de FIN est postérieure à aujourd'hui
source_data = source_data[
    source_data["lastdate_end"] >= current_date
]

print(f'Nombre de lignes après filtrage des événements futurs : {source_data.shape[0]}')

# Vérification : afficher quelques dates pour contrôle
print("\n=== Vérification des dates ===")
print(source_data[["uid", "firstdate_begin", "lastdate_end"]].head(10))

Date actuelle de référence : 2026-05-12 03:35:32.773627+00:00
Nombre de lignes après filtrage des événements futurs : 2081

=== Vérification des dates ===
         uid           firstdate_begin              lastdate_end
57  47780736 2026-05-16 08:00:00+00:00 2026-05-16 16:00:00+00:00
65  99876723 2026-06-06 06:00:00+00:00 2026-06-06 10:00:00+00:00
66  36187635 2026-06-06 07:00:00+00:00 2026-06-06 15:30:00+00:00
67  69844964 2026-06-05 08:00:00+00:00 2026-06-07 16:00:00+00:00
68  84771205 2026-06-03 12:00:00+00:00 2026-06-03 14:00:00+00:00
69  53900658 2026-05-23 15:00:00+00:00 2026-05-23 21:59:00+00:00
70  86224351 2026-05-23 15:00:00+00:00 2026-05-23 21:59:00+00:00
71  56887708 2026-05-31 18:00:00+00:00 2026-05-31 19:00:00+00:00
72  31492252 2026-07-03 18:00:00+00:00 2026-07-03 20:30:00+00:00
73  90146175 2026-06-13 12:30:00+00:00 2026-06-13 14:00:00+00:00


In [15]:
# Ajouter cette cellule après le filtrage pour vérifier
print("\n=== VÉRIFICATION DES DATES ===")
current_date = pd.Timestamp.now(tz='UTC')

# Vérifier s'il reste des événements passés
past_events = source_data[
    pd.to_datetime(source_data["lastdate_end"], utc=True) < current_date
]

if len(past_events) > 0:
    print(f"⚠️ ATTENTION : {len(past_events)} événements passés détectés !")
    print(past_events[["uid", "firstdate_begin", "lastdate_end"]].head())
else:
    print(f"✓ Tous les {len(source_data)} événements sont futurs")

# Statistiques temporelles
print(f"\nDate la plus proche : {source_data['firstdate_begin'].min()}")
print(f"Date la plus lointaine : {source_data['lastdate_end'].max()}")



=== VÉRIFICATION DES DATES ===
✓ Tous les 2081 événements sont futurs

Date la plus proche : 2022-06-14 07:00:00+00:00
Date la plus lointaine : 2028-07-28 16:00:00+00:00


In [16]:
# List all event on july 2026
july_2026_events = source_data[source_data["firstdate_begin"].dt.month == 7]
july_2026_events = july_2026_events[july_2026_events["firstdate_begin"].dt.year == 2026]
print(f"\nÉvénements de juillet 2026 : {len(july_2026_events)}")
print(july_2026_events[["uid", "firstdate_begin", "lastdate_end"]])



Événements de juillet 2026 : 143
           uid           firstdate_begin              lastdate_end
72    31492252 2026-07-03 18:00:00+00:00 2026-07-03 20:30:00+00:00
97    25918870 2026-07-30 11:45:00+00:00 2026-07-30 13:30:00+00:00
206    1507761 2026-07-23 12:00:00+00:00 2026-08-06 12:30:00+00:00
210   12189212 2026-07-24 18:30:00+00:00 2026-07-24 21:00:00+00:00
333   34119007 2026-07-04 15:00:00+00:00 2026-07-04 20:00:00+00:00
...        ...                       ...                       ...
6629   8540067 2026-07-07 07:00:00+00:00 2026-07-07 10:00:00+00:00
6717  19169273 2026-07-24 18:30:00+00:00 2026-07-24 20:30:00+00:00
6721  18973857 2026-07-03 17:00:00+00:00 2026-07-03 21:00:00+00:00
6796  69972996 2026-07-16 11:30:00+00:00 2026-07-16 13:00:00+00:00
6833  42958907 2026-07-23 09:00:00+00:00 2026-07-23 10:00:00+00:00

[143 rows x 3 columns]


In [6]:
# Analyser vos descriptions
source_data["token_count"] = source_data["longdescription_fr"].apply(count_tokens)

# Statistiques
print(f"Nombre moyen de tokens: {source_data['token_count'].mean():.2f}")
print(f"Nombre max de tokens: {source_data['token_count'].max()}")
print(f"Nombre min de tokens: {source_data['token_count'].min()}")

# Afficher la distribution
print("\nDistribution des tokens:")
print(source_data["token_count"].describe())

# Voir les descriptions les plus longues
print("\nTop 5 descriptions les plus longues:")
print(source_data.nlargest(5, "token_count")[["uid", "longdescription_fr", "token_count"]])

Nombre moyen de tokens: 171.36
Nombre max de tokens: 2478
Nombre min de tokens: 11

Distribution des tokens:
count    2090.000000
mean      171.362679
std       183.460551
min        11.000000
25%        86.000000
50%       133.000000
75%       159.000000
max      2478.000000
Name: token_count, dtype: float64

Top 5 descriptions les plus longues:
           uid                                 longdescription_fr  token_count
3939  90961540  <h2>Valérie Jouve, (en sa présence) : <em>Les ...         2478
5556  15644546  <h2>Performance de Mouna Ahizoune, <em>When i ...         2409
6758  12339413  <p>Enfermée dans un camp du simple fait de ses...         1977
3040  24200773  <p><em>Une invitation de Lieux Publics à inves...         1677
2779  46935279  <p><strong>Les papillons de nuit</strong><br><...         1614


## Chunking

Après analyse des données, le constat est le suivant : 
- 75% des descriptions < 153 tokens → Pas besoin de chunking
- Max 1983 tokens → Quelques descriptions très longues nécessitent un découpage
- Moyenne 151 tokens → Largement sous la limite de 512 tokens

Conclusion, nous allons utiliser le chunking conditionnel en découpant uniquement les descriptions qui sont supérieures à 512 tokens.

In [7]:
# Configuration du splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # Taille cible en tokens (marge de sécurité)
    chunk_overlap=50,  # Chevauchement pour garder le contexte
    length_function=count_tokens,
    separators=["\n\n", "\n", ". ", " ", ""]  # Découpe intelligente
)

def process_description(row):
    """Découpe uniquement si nécessaire"""
    text = row["longdescription_fr"]
    token_count = row["token_count"]
    uid = row["uid"]
    firstdate_begin = row["firstdate_begin"]
    firstdate_end = row["firstdate_end"]
    lastdate_begin = row["lastdate_begin"]
    lastdate_end = row["lastdate_end"]
    
    if token_count <= 512:
        # Pas de chunking nécessaire
        return [{
            "uid": uid,
            "chunk_id": 0,
            "text": text,
            "token_count": token_count,
            "firstdate_begin": firstdate_begin,
            "firstdate_end": firstdate_end,
            "lastdate_begin": lastdate_begin,
            "lastdate_end": lastdate_end
        }]
    else:
        # Chunking pour les longues descriptions
        chunks = text_splitter.split_text(text)
        return [{
            "uid": uid,
            "chunk_id": i,
            "text": chunk,
            "token_count": count_tokens(chunk),
            "firstdate_begin": firstdate_begin,
            "firstdate_end": firstdate_end,
            "lastdate_begin": lastdate_begin,
            "lastdate_end": lastdate_end
        } for i, chunk in enumerate(chunks)]

# Appliquer le traitement
chunks_list = []
for _, row in source_data.iterrows():
    chunks_list.extend(process_description(row))

# Créer un nouveau DataFrame avec les chunks
chunks_df = pd.DataFrame(chunks_list)

print(f"Nombre de descriptions originales: {len(source_data)}")
print(f"Nombre total de chunks: {len(chunks_df)}")
print(f"Descriptions découpées: {len(chunks_df[chunks_df['chunk_id'] > 0]['uid'].unique())}")

Nombre de descriptions originales: 2090
Nombre total de chunks: 2282
Descriptions découpées: 97


## Embedding
Utilisation de Mistral pour l'embedding

In [8]:
def create_smart_batches(texts, max_tokens_per_batch=8000):
    """Crée des batches en fonction du nombre de tokens"""
    batches = []
    current_batch = []
    current_tokens = 0
    
    for text in texts:
        text_tokens = count_tokens(text)
        
        if current_tokens + text_tokens > max_tokens_per_batch and current_batch:
            batches.append(current_batch)
            current_batch = [text]
            current_tokens = text_tokens
        else:
            current_batch.append(text)
            current_tokens += text_tokens
    
    if current_batch:
        batches.append(current_batch)
    
    return batches

In [9]:
texts_to_embed = chunks_df["text"].tolist()

batches = create_smart_batches(texts_to_embed, max_tokens_per_batch=7000)
print(f"Nombre de batches créés: {len(batches)}")

all_embeddings = []

with Mistral(
    api_key=os.getenv("MISTRAL_API_KEY", ""),
) as mistral:
    
    for i, batch in enumerate(batches):
        print(f"Batch {i+1}/{len(batches)} - {len(batch)} textes")
        
        try:
            res = mistral.embeddings.create(
                model="mistral-embed", 
                inputs=batch
            )
            
            all_embeddings.extend([item.embedding for item in res.data])
            print(f"Total embeddings accumulés: {len(all_embeddings)}")
            
            # Pause entre les batches pour respecter le rate limit
            if i < len(batches) - 1:  # Pas de pause après le dernier batch
                wait_time = 2  # Secondes entre chaque batch
                print(f"Pause de {wait_time}s...")
                time.sleep(wait_time)
                
        except Exception as e:
            if "429" in str(e) or "rate_limited" in str(e).lower():
                print(f"Rate limit atteint, pause de 60s...")
                time.sleep(60)
                # Réessayer le batch
                print(f"Nouvelle tentative pour le batch {i+1}")
                res = mistral.embeddings.create(
                    model="mistral-embed", 
                    inputs=batch
                )
                all_embeddings.extend([item.embedding for item in res.data])
                print(f"  ✓ Batch traité après retry")
            else:
                print(f"  ✗ Erreur: {e}")
                raise

# Vérification avant assignation
print(f"\nVérification:")
print(f"  - Nombre de chunks: {len(chunks_df)}")
print(f"  - Nombre d'embeddings: {len(all_embeddings)}")

# Ajouter les embeddings au DataFrame de chunks
chunks_df["embedding"] = all_embeddings

# Sauvegarder les métadonnées pour la recherche
chunks_df.to_json("chunks_with_embeddings.json", orient="records", force_ascii=False)

Nombre de batches créés: 53
Batch 1/53 - 51 textes
Total embeddings accumulés: 51
Pause de 2s...
Batch 2/53 - 50 textes
Total embeddings accumulés: 101
Pause de 2s...
Batch 3/53 - 56 textes
Total embeddings accumulés: 157
Pause de 2s...
Batch 4/53 - 30 textes
Total embeddings accumulés: 187
Pause de 2s...
Batch 5/53 - 39 textes
Total embeddings accumulés: 226
Pause de 2s...
Batch 6/53 - 37 textes
Total embeddings accumulés: 263
Pause de 2s...
Batch 7/53 - 42 textes
Total embeddings accumulés: 305
Pause de 2s...
Batch 8/53 - 53 textes
Total embeddings accumulés: 358
Pause de 2s...
Batch 9/53 - 36 textes
Total embeddings accumulés: 394
Pause de 2s...
Batch 10/53 - 49 textes
Total embeddings accumulés: 443
Pause de 2s...
Batch 11/53 - 41 textes
Total embeddings accumulés: 484
Pause de 2s...
Batch 12/53 - 49 textes
Total embeddings accumulés: 533
Pause de 2s...
Batch 13/53 - 45 textes
Total embeddings accumulés: 578
Pause de 2s...
Batch 14/53 - 40 textes
Total embeddings accumulés: 618
Pau

In [10]:
# Initialisation de l'index Faiss
embeddings_array = np.array([item for item in chunks_df["embedding"]])
dimension = embeddings_array.shape[1]  # Nombre de features
n_vectors = embeddings_array.shape[0]
print(f"Dimension des vecteurs: {dimension}")
print(f"Nombre de vecteurs à indexer: {n_vectors}")

# OPTIMISATION: Choix de l'index selon la taille
if n_vectors < 10000:
    # Pour petits datasets: IndexFlatL2 (exact, rapide)
    index = faiss.IndexFlatL2(dimension)
    print("Index utilisé: IndexFlatL2 (recherche exacte)")
else:
    # Pour grands datasets: IndexIVFFlat (approximatif, plus rapide)
    nlist = min(100, n_vectors // 39)  # Nombre de clusters
    quantizer = faiss.IndexFlatL2(dimension)
    index = faiss.IndexIVFFlat(quantizer, dimension, nlist)
    
    # Entraînement de l'index
    print(f"Entraînement de l'index avec {nlist} clusters...")
    index.train(embeddings_array)
    print("Index utilisé: IndexIVFFlat (recherche approximative rapide)")

# Ajout des embeddings à l'index
index.add(embeddings_array)

# VÉRIFICATION 2: Tous les vecteurs sont dans l'index
assert index.ntotal == n_vectors, f"Index incomplet! {index.ntotal}/{n_vectors}"
print(f"Vérification: {index.ntotal} vecteurs indexés sur {n_vectors}")

# Sauvegarde de l'index sur le disque
faiss.write_index(index, "faiss_index.idx")

# VÉRIFICATION 3: Test de cohérence
print("\n=== Test de cohérence ===")
test_vector = embeddings_array[0:1]
distances, indices = index.search(test_vector, 1)
assert indices[0][0] == 0, "L'index ne retourne pas le bon vecteur!"
print("Test de cohérence réussi")

# Statistiques finales
print(f"\n=== Statistiques ===")
print(f"Événements originaux: {len(source_data)}")
print(f"Chunks créés: {len(chunks_df)}")
print(f"Vecteurs indexés: {index.ntotal}")
print(f"Taux de couverture: {(index.ntotal / len(source_data)) * 100:.1f}%")

Dimension des vecteurs: 1024
Nombre de vecteurs à indexer: 2282
Index utilisé: IndexFlatL2 (recherche exacte)
Vérification: 2282 vecteurs indexés sur 2282

=== Test de cohérence ===
Test de cohérence réussi

=== Statistiques ===
Événements originaux: 2090
Chunks créés: 2282
Vecteurs indexés: 2282
Taux de couverture: 109.2%


In [11]:
# VÉRIFICATION 4: Distribution des distances
print("\n=== Analyse de la qualité de l'index ===")

# Recherche de tous les vecteurs dans l'index
all_distances, all_indices = index.search(embeddings_array[:100], 1)

print(f"Distance moyenne à soi-même: {all_distances.mean():.6f}")
print(f"Distance max à soi-même: {all_distances.max():.6f}")

if all_distances.mean() > 0.001:
    print("Attention: Les vecteurs ne se retrouvent pas exactement")
else:
    print("Qualité de l'index: Excellente")

# VÉRIFICATION 5: Couverture des événements originaux
indexed_uids = set(chunks_df['uid'].unique())
original_uids = set(source_data['uid'].unique())

missing_uids = original_uids - indexed_uids
if missing_uids:
    print(f"{len(missing_uids)} événements manquants: {list(missing_uids)[:5]}")
else:
    print(f"Tous les {len(original_uids)} événements sont indexés")



=== Analyse de la qualité de l'index ===
Distance moyenne à soi-même: 0.000000
Distance max à soi-même: 0.000000
Qualité de l'index: Excellente
Tous les 2090 événements sont indexés


In [ ]:
# Exemple de recherche
def search_events(query, k=5):
    """Recherche les k événements les plus similaires"""
    
    # 1. Vectoriser la requête
    with Mistral(api_key=os.getenv("MISTRAL_API_KEY", "")) as mistral:
        query_res = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[query]
        )
        query_embedding = np.array([query_res.data[0].embedding])
    
    # 2. Rechercher dans FAISS
    distances, indices = index.search(query_embedding, k)
    
    # 3. Récupérer les résultats
    results = chunks_df.iloc[indices[0]]
    
    return results[["uid", "chunk_id", "text", "token_count"]], distances[0]

# Test
results, distances = search_events("concert de musique classique", k=5)
print("Top 5 événements similaires:")
print(results)
print(f"\nDistances: {distances}")